In [ ]:
if 'google.colab' in str(get_ipython()):
    ! git clone -b main https://github.com/edsonportosilva/OptiCommPy
    from os import chdir as cd
    cd('/content/OptiCommPy/')
    ! pip install .

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
from numpy.fft import fft, ifft, fftfreq
from  scipy.constants import c
import matplotlib.pyplot as plt

from optic.comm.modulation import modulateGray, grayMapping
from optic.dsp.core import pnorm, upsample, firFilter, pulseShape, signal_power, phaseNoise, decimate, symbolSync
from optic.dsp.equalization import edc, mimoAdaptEqualizer
from optic.models.devices import iqm, coherentReceiver, pdmCoherentReceiver, basicLaserModel, mzm, edfa
from optic.models.channels import linearFiberChannel, awgn
from optic.utils import parameters
from optic.plot import pconst, eyediagram, plotPSD
from optic.models.tx import simpleWDMTx
from optic.dsp.carrierRecovery import cpr

## Equalização
---

## Sumário
---

- [Equalizador MIMO 2 x 2](#equalizador-MIMO-2-x-2)


### Equalizador MIMO 2 x 2

Em sistemas coerentes modernos, dois sinais independentes são transmitidos simultaneamente em duas polarizações ortogonais (X e Y, ou V e H), dobrando a taxa de dados sem aumento de largura de banda — isso é PDM (Polarization-Division Multiplexing).

No entanto, devido à dispersão de modo de polarização (PMD), as polarizações se misturam durante a propagação na fibra. Quando o sinal chega no receptor, não está mais separado como foi transmitido.



<center><img src = "https://github.com/silasabs/CoherentOptics/blob/main/figures/mimo%20butterfly%20equalizer.png?raw=True"/></center>
<center>GitHub Silas João</center>

O equalizador MIMO 2×2 reconstrói os sinais transmitidos separando corretamente os dois fluxos de dados da mistura recebida, usando quatro filtros FIR adaptativos:

$$\begin{bmatrix}
y1[k] \\ y2[k]
\end{bmatrix} =
\begin{bmatrix}
W_{1V}[n] & W_{1H}[n] \\
W_{2V}[n] & W_{2H}[n]
\end{bmatrix} .
\begin{bmatrix}
X_V[k] \\ X_H[k]
\end{bmatrix}$$

  - $X_V[k]:$ entrada da polarização vertical
  - $X_H[k]:$ entrada da polarização horizontal
  - $y_1[k],y_2[k]:$ sinais equalizados que idealmente correspondem aos sinais transmitidos

Esse modelo é chamado de "butterfly equalizer" pela forma de cruzamento dos filtros na arquitetura.

O uso do equalizador MIMO 2x2 (butterfly) é imprescindível em sistemas coerentes com multiplexação por polarização, pois ele permite separar sinais misturados pela fibra (PMD), restaurar as constelações originais e viabilizar altas taxas de transmissão. É um dos principais blocos que você deve dominar para avançar seu projeto de compensação de PMD com equalizadores adaptativos.

In [ ]:
def ddrlsUp(x, constSymb, nModes, paramEq):
    """
    Decision-Directed RLS Equalizer (MIMO 2x2)

    Returns
    -------
    tuple
        - y (np.array): estimativa dos símbolos.
        - e (np.array): erro associado a cada modo de polarização.
        - w (np.array): matriz de coeficientes.
    """
    N = len(x) - paramEq.nTaps + 1
    delay = (paramEq.nTaps - 1) // 2

    y = np.zeros((len(x), nModes), dtype='complex')
    e = np.zeros((len(x), nModes), dtype='complex')
    w = np.zeros((paramEq.nTaps, nModes**2), dtype='complex')
    P = [np.eye(paramEq.nTaps, dtype='complex') / 1e-2 for _ in range(4)]
    w[:,0][delay] = 1

    for n in tqdm(range(N), disable=not (paramEq.progBar)):
        xH = np.flipud(x[:,0][n:n+paramEq.nTaps])
        xV = np.flipud(x[:,1][n:n+paramEq.nTaps])

        y[:,0][n] = np.dot(w[:,0], xV) + np.dot(w[:,1], xH)
        y[:,1][n] = np.dot(w[:,2], xV) + np.dot(w[:,3], xH)

        d0 = constSymb[np.argmin(np.abs(y[:,0][n] - constSymb))]
        d1 = constSymb[np.argmin(np.abs(y[:,1][n] - constSymb))]

        e[:,0][n] = d0 - y[:,0][n]
        e[:,1][n] = d1 - y[:,1][n]

        for idx, (xi, err, Pi, wi) in enumerate(zip([xV, xH]*2, [e[:,0][n]]*2 + [e[:,1][n]]*2, P, [0, 1, 2, 3])):
            k = (Pi @ xi) / (paramEq.lambdaRLS + xi.conj().T @ Pi @ xi)
            P[wi] = (Pi - np.outer(k, xi.conj().T @ Pi)) / paramEq.lambdaRLS
            w[:,wi] += k * err

    return y, e, w

In [ ]:
def ddlmsUp(x, constSymb, nModes, paramEq):
    """
    Decision-Directed LMS Equalizer (MIMO 2x2)

    Returns
    -------
    tuple
        - y (np.array): estimativa dos símbolos.
        - e (np.array): erro associado a cada modo de polarização.
        - w (np.array): matriz de coeficientes.
    """
    N = len(x) - paramEq.nTaps + 1
    delay = (paramEq.nTaps - 1) // 2

    y = np.zeros((len(x), nModes), dtype='complex')
    e = np.zeros((len(x), nModes), dtype='complex')
    w = np.zeros((paramEq.nTaps, nModes**2), dtype='complex')
    w[:,0][delay] = 1

    for n in tqdm(range(N), disable=not (paramEq.progBar)):
        xH = np.flipud(x[:,0][n:n+paramEq.nTaps])
        xV = np.flipud(x[:,1][n:n+paramEq.nTaps])

        y[:,0][n] = np.dot(w[:,0], xV) + np.dot(w[:,1], xH)
        y[:,1][n] = np.dot(w[:,2], xV) + np.dot(w[:,3], xH)

        # decisão brusca
        d0 = constSymb[np.argmin(np.abs(y[:,0][n] - constSymb))]
        d1 = constSymb[np.argmin(np.abs(y[:,1][n] - constSymb))]

        e[:,0][n] = d0 - y[:,0][n]
        e[:,1][n] = d1 - y[:,1][n]

        w[:,0] += paramEq.mu[0] * np.conj(xV) * e[:,0][n]
        w[:,2] += paramEq.mu[0] * np.conj(xV) * e[:,1][n]
        w[:,1] += paramEq.mu[0] * np.conj(xH) * e[:,0][n]
        w[:,3] += paramEq.mu[0] * np.conj(xH) * e[:,1][n]

    return y, e, w

In [ ]:
def rdeUp(x, constSymb, nModes, paramEq, y=None, e=None, w=None, preConv=False):
    """
    Radius-Directed Equalization Algorithm

    Returns
    -------
    tuple
        - y (np.array): estimativa dos símbolos.
        - e (np.array): erro associado a cada modo de polarização.
        - w (np.array): matriz de coeficientes.

    Referências
    -----------
    [1] Digital Coherent Optical Systems, Architecture and Algorithms
    """

    N = len(x) - paramEq.nTaps + 1

    # Obtém o atraso da filtragem FIR
    delay = (paramEq.nTaps - 1) // 2

    # obtem os raios da constelação M-QAM
    Rrde = np.unique(np.abs(constSymb))

    if preConv == False:

        paramEq.N2 = 0

        y = np.zeros((len(x), nModes), dtype='complex')
        e = np.zeros((len(x), nModes), dtype='complex')
        w = np.zeros((paramEq.nTaps, nModes**2), dtype='complex')

        # single spike initialization
        w[:,0][delay] = 1

    for n in tqdm(range(paramEq.N2, N), disable=not (paramEq.progBar)):

        xH = np.flipud(x[:,0][n:n+paramEq.nTaps])
        xV = np.flipud(x[:,1][n:n+paramEq.nTaps])

        # calcula a saída do equalizador 2x2
        y[:,0][n] = np.dot(w[:,0], xV) + np.dot(w[:,1], xH)
        y[:,1][n] = np.dot(w[:,2], xV) + np.dot(w[:,3], xH)

        R1 = np.argmin(np.abs(Rrde - np.abs(y[:,0][n])))
        R2 = np.argmin(np.abs(Rrde - np.abs(y[:,1][n])))

        # calcula e atualiza erro para cada modo de polarização
        e[:,0][n] = y[:,0][n] * (Rrde[R1]**2 - np.abs(y[:,0][n])**2)
        e[:,1][n] = y[:,1][n] * (Rrde[R2]**2 - np.abs(y[:,1][n])**2)

        # atualiza os coeficientes do equalizador
        w[:,0] += paramEq.mu[0] * np.conj(xV) * e[:,0][n]
        w[:,2] += paramEq.mu[0] * np.conj(xV) * e[:,1][n]
        w[:,1] += paramEq.mu[0] * np.conj(xH) * e[:,0][n]
        w[:,3] += paramEq.mu[0] * np.conj(xH) * e[:,1][n]

        if n == paramEq.N1:
            # Defina a polarização Y como ortogonal a X para evitar
            # a convergência para a mesma polarização (evitar a singularidade CMA)
            w[:,3] =  np.conj(w[:,0][::-1])
            w[:,2] = -np.conj(w[:,1][::-1])

    return y, e, w

In [ ]:
def cmaUp(x, constSymb, nModes, paramEq, preConv=False):
    """
    Constant-Modulus Algorithm

    Returns
    -------
    tuple
        - y (np.array): estimativa dos símbolos.
        - e (np.array): erro associado a cada modo de polarização.
        - w (np.array): matriz de coeficientes.

    Referências
    -----------
    [1] Digital Coherent Optical Systems, Architecture and Algorithms
    """

    N = len(x) - paramEq.nTaps + 1

    # Obtém o atraso da filtragem FIR
    delay = (paramEq.nTaps - 1) // 2

    y = np.zeros((len(x), nModes), dtype='complex')
    e = np.zeros((len(x), nModes), dtype='complex')
    w = np.zeros((paramEq.nTaps, nModes**2), dtype='complex')

    # single spike initialization
    w[:,0][delay] = 1

    # constante relacionada às características da modulação para o algoritmo CMA
    R = np.mean(np.abs(constSymb)**4) / np.mean(np.abs(constSymb)**2)

    for n in tqdm(range(N), disable=not (paramEq.progBar)):

        xH = np.flipud(x[:,0][n:n+paramEq.nTaps])
        xV = np.flipud(x[:,1][n:n+paramEq.nTaps])

        # calcula a saída do equalizador 2x2
        y[:,0][n] = np.dot(w[:,0], xV) + np.dot(w[:,1], xH)
        y[:,1][n] = np.dot(w[:,2], xV) + np.dot(w[:,3], xH)

        # calcula e atualiza erro para cada modo de polarização
        e[:,0][n] = y[:,0][n] * (R - np.abs(y[:,0][n])**2)
        e[:,1][n] = y[:,1][n] * (R - np.abs(y[:,1][n])**2)

        # atualiza os coeficientes do filtro
        w[:,0] += paramEq.mu[0] * np.conj(xV) * e[:,0][n]
        w[:,2] += paramEq.mu[0] * np.conj(xV) * e[:,1][n]
        w[:,1] += paramEq.mu[0] * np.conj(xH) * e[:,0][n]
        w[:,3] += paramEq.mu[0] * np.conj(xH) * e[:,1][n]

        if n == paramEq.N1:
            # Defina a polarização Y como ortogonal a X para evitar
            # a convergência para a mesma polarização (evitar a singularidade CMA)
            w[:,3] =  np.conj(w[:,0][::-1])
            w[:,2] = -np.conj(w[:,1][::-1])

        if preConv and n == paramEq.N2:
            break

    if preConv:
        w = w/np.max(np.abs(w))
        y, e, w = rdeUp(x, constSymb, nModes, paramEq, y, e, w, preConv=True)

    return y, e, w

In [ ]:
def mimoAdaptEq(x, paramEq):
    """
    Equalizador adaptativo MIMO 2x2

    Returns
    -------
    tuple
        - y (np.array): estimativa dos símbolos.
        - e (np.array): erro associado a cada modo de polarização.
        - w (np.array): matriz de coeficientes.

    Raises
    ------
    ValueError
        Caso o sinal não possua duas polarizações.

    ValueError
        Caso o algoritmo seja especificado incorretamente.
    """

    if x.shape[1] != 2:
        raise ValueError("O sinal deve conter duas polarizações")

    nModes = x.shape[1]

    # obtem os símbolos da constelação
    constSymb = grayMapping(paramEq.M, paramEq.constType)
    # normaliza os símbolos da constelação
    constSymb = pnorm(constSymb)

    if paramEq.alg == 'cma':
        y, e, w = cmaUp(x, constSymb, nModes, paramEq)
    elif paramEq.alg == 'rde':
        y, e, w = rdeUp(x, constSymb, nModes, paramEq)
    elif paramEq.alg == 'cma-to-rde':
        y, e, w = cmaUp(x, constSymb, nModes, paramEq, preConv=True)
    elif paramEq.alg == 'dd-lms':
        y, e, w = ddlmsUp(x, constSymb, nModes, paramEq)
    elif paramEq.alg == 'dd-rls':
        y, e, w = ddrlsUp(x, constSymb, nModes, paramEq)
    else:
        raise ValueError("Algoritmo de equalização especificado incorretamente.")

    return y, e, w

### Simulação WDM 32G PM-64QAM

In [ ]:
##################### TRANSMISSOR #####################

# Transmitter parameters:
paramTx = parameters()
paramTx.M = 64                                # order of the modulation format
paramTx.constType = 'qam'                     # modulation scheme
paramTx.Rs  = 32e9                            # symbol rate [baud]
paramTx.SpS = 16                              # samples per symbol
paramTx.pulse = 'rrc'                         # pulse shaping filterN
paramTx.Ntaps = 4096                          # number of pulse shaping filter coefficients
paramTx.alphaRRC = 0.01                       # RRC rolloff
paramTx.Pch_dBm = -2                          # power per WDM channel [dBm]
paramTx.Nch     = 1                           # number of WDM channels
paramTx.Fc      = 193.1e12                    # central optical frequency of the WDM spectrum
paramTx.lw      = 100e3                       # laser linewidth in Hz
paramTx.freqSpac = 37.5e9                     # WDM grid spacing
paramTx.Nmodes =  2                           # number of signal modes [2 for polarization multiplexed signals]
paramTx.Nbits = int(np.log2(paramTx.M)*2e5)   # total number of bits per polarization

Fs = paramTx.Rs * paramTx.SpS                 # simulation sampling rate

## fiber parameters:
paramFiber = parameters()
paramFiber.L = 300                            # comprimento do enlace [km]
paramFiber.alpha = 0.0                        # coeficiente de perdas [dB/Km]
paramFiber.D = 17                             # parâmetro de dispersão [ps/nm/km]
paramFiber.Fs = Fs                            # Frequência de amostragem do sinal [amostras/segundo]

# generate WDM signal
sigWDM_Tx, symbTx_, paramTx = simpleWDMTx(paramTx)



In [ ]:
paramEDFA = parameters()
paramEDFA.G = paramFiber.alpha * paramFiber.L # Ganho para compensar
paramEDFA.Fs = Fs
paramEDFA.NF = 7
paramEDFA.Fc = paramTx.Fc
#sigCh = edfa(sigWDM_Tx, paramEDFA)            # modelo linear do EDFA

# # deteriorates the signal-to-noise ratio before the fiber
#SNR = 13
#sigAWGN = awgn(sigWDM_Tx, SNR, Fs, paramTx.Rs)

# simulate linear signal propagation
sigCh = linearFiberChannel(sigWDM_Tx, paramFiber)

# plot psd
fig,_ = plotPSD(sigWDM_Tx, Fs, paramFiber.Fc, label='Tx');
fig, ax = plotPSD(sigCh, Fs, paramFiber.Fc, fig=fig, label='Rx');
fig.set_figwidth(8)
ax.set_title('optical WDM spectrum');
print('P_sig[opt] = %.2f dBm'%(10*np.log10(signal_power(sigCh)/1e-3)))

In [ ]:
# parameters
chIndex = 0  # index of the channel to be demodulated

Fc = paramTx.Fc
Ts = 1/Fs
freqGrid = paramTx.freqGrid
π  = np.pi
t  = np.arange(0, len(sigCh))*Ts

print('Demodulating channel #%d , fc: %.4f THz, λ: %.4f nm\n'\
      %(chIndex, (Fc + freqGrid[chIndex])/1e12, c/(Fc + freqGrid[chIndex])/1e-9))

symbTx = symbTx_[:,:,chIndex]

# local oscillator (LO) parameters:
FO      = 150e6                 # frequency offset
Δf_lo   = freqGrid[chIndex]+FO  # downshift of the channel to be demodulated

# generate CW laser LO field
paramLO = parameters()
paramLO.P = 10              # power in dBm
paramLO.lw = 100e3          # laser linewidth
paramLO.RIN_var = 0
paramLO.Ns = len(sigCh)
paramLO.Fs = Fs

sigLO = basicLaserModel(paramLO)
sigLO = sigLO*np.exp(1j*2*π*Δf_lo*t) # add frequency offset

print('Local oscillator P: %.2f dBm, lw: %.2f kHz, FO: %.2f MHz\n'\
      %(paramLO.P, paramLO.lw/1e3, FO/1e6))

# polarization multiplexed coherent optical receiver

# photodiodes parameters
paramPD = parameters()
paramPD.B = paramTx.Rs
paramPD.Fs = Fs
paramPD.ideal = True

# sigRx_ = coherentReceiver(sigCh, sigLO)

θsig = π/3 # polarization rotation angle
sigRx = pdmCoherentReceiver(sigCh, sigLO, θsig, paramPD)

# SNR = 25
# sigRx = awgn(sigRx, SNR, Fs, paramTx.Rs)

# plot received constellations
pconst(sigRx[0::paramTx.SpS], R=3);
print('P_sig[opt] = %.2f dBm'%(10*np.log10(signal_power(sigRx)/1e-3)))

In [ ]:
# Matched filtering
if paramTx.pulse == 'nrz':
    pulse = pulseShape('nrz', paramTx.SpS)
elif paramTx.pulse == 'rrc':
    pulse = pulseShape('rrc', paramTx.SpS, N=paramTx.Ntaps, alpha=paramTx.alphaRRC, Ts=1/paramTx.Rs)

pulse = pnorm(pulse)
sigRx = firFilter(pulse, sigRx)

# plot constellations after matched filtering
pconst(sigRx[0::paramTx.SpS,:], R=3)

# CD compensation
paramEDC = parameters()
paramEDC.L = paramFiber.L
paramEDC.D = paramFiber.D
paramEDC.Fc = Fc-Δf_lo
paramEDC.Fs = Fs

sigRx = edc(sigRx, paramEDC)
pconst(sigRx[0::paramTx.SpS,:], R=3);
print('P_sig[opt] = %.2f dBm'%(10*np.log10(signal_power(sigRx)/1e-3)))

In [ ]:
# decimate to one sample per symbol
paramDec = parameters()
paramDec.SpS_in  = paramTx.SpS
paramDec.SpS_out = 1
sigRx = decimate(sigRx, paramDec)

x = pnorm(sigRx)
print('P_sig[opt] = %.2f dBm'%(10*np.log10(signal_power(x)/1e-3)))

In [ ]:

# parâmetros do equalizador
paramEq = parameters()
paramEq.nTaps = 7
paramEq.mu = [5e-3]
paramEq.M = paramTx.M
paramEq.constType = paramTx.constType
paramEq.N1   = 4000
paramEq.N2   = 8500
paramEq.alg = 'cma-to-rde'
paramEq.progBar = True


y_EQ = mimoAdaptEq(x, paramEq)
discard = int(0.1 * y_EQ[0].shape[0])
pconst(y_EQ[0][discard:-discard,:], R=2);

# Equalizador Adaptativo
paramEq.alg = 'dd-lms'
y_LMS, e, _ = mimoAdaptEq(y_EQ[0], paramEq)

discard = 20000
pconst(y_LMS[discard:-discard,:], R=2);


fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].plot(np.abs(y_EQ[1][:, 0]), label='Mode 0')
axs[0].set_ylabel(r'$|e{[n]}|$')
axs[0].set_xlabel(r'$Iterações$')
axs[0].legend()
axs[0].grid(True)

axs[1].plot(np.abs(y_EQ[1][:, 1]), label='Mode 1')
axs[1].set_ylabel(r'$|e{[n]}|$')
axs[1].set_xlabel(r'$Iterações$')
axs[1].legend()
axs[1].grid(True)

In [ ]:
paramCPR = parameters()
paramCPR.alg = 'bps'
paramCPR.M   = paramTx.M
paramCPR.N   = 75
paramCPR.B   = 64

y_CPR     = cpr(y_EQ[0], paramCPR)
y_CPR_RDE = cpr(y_LMS, paramCPR)

discard = int(0.1 * y_CPR.shape[0])
y = y_CPR[discard:-discard,:]
mask_valid = np.all(np.abs(y) > 1e-12, axis=1)
y = y[mask_valid]

#plot constellations after CPR
pconst(y_CPR[discard:-discard,:]);

print('\nEqualização sem pre-convergência')
pconst(y_CPR_RDE[discard:-discard,:]);

In [ ]:
def plotUtils(x, d, title):
    """
    Plote sinais com dois modos de polarização em conjunto
    com os símbolos do transmissor.

    Parameters
    ----------
    x : np.array
        Sinal de entrada com dois modos de polarização.

    d : np.array
        Símbolos do transmissor.

    title : string
        Titulo desejado.
    """

    if x.shape[1] != 2:
        raise ValueError('os sinais devem possuir dois modos de polarização')

    plt.plot(np.abs(x[:,0]), '.', color='black',  label = 'After CPR Mode 0')
    plt.plot(np.abs(x[:,1]), '.', color='purple', label = 'After CPR Mode 1')
    plt.plot(np.abs(d[:,0]), '.', color='orange', label = 'SymbTx')

    plt.title(title)
    plt.ylabel(r'$|y_{1,2}[k]|$')
    plt.xlabel(r'$k$')
    plt.ylim(0, 2)

    plt.legend(loc='upper center')
    plt.grid()

    plt.show()

In [ ]:
plotUtils(y_CPR, symbTx, 'CMA-to-RDE')

In [ ]:
plotUtils(y_CPR_RDE, symbTx, 'RDE')

In [ ]:
def estimate_snr_ber(sigRx, symbTx, M=16, constType='qam', discard=2000, SpS=1):
    """
    Calcula SNR e BER entre sinal recebido e transmitido.
    # Seleciona amostras centrais se houver oversampling

    Retorna
    -------
    SNR_dB : float
        Relação sinal-ruído estimada em dB.
    BER : float
        Taxa de erro de bits.
    """
    def symb_to_bits(idx, M):
        """Converte índices da constelação para bits binários"""
        k = int(np.log2(M))
        bits = ((idx[:,None] & (1 << np.arange(k)[::-1])) > 0).astype(int)
        return bits.reshape(-1)

    def qam_demod(rx, constSymb):
        """Faz decisão dura com base na distância mínima à constelação"""
        dists = np.abs(rx.reshape(-1,1) - constSymb.reshape(1,-1))**2
        idx = np.argmin(dists, axis=1)
        return idx

    if SpS > 1:
        sigRx = sigRx[SpS//2::SpS]

    # Normalização da constelação
    constSymb = grayMapping(M, constType)
    symbTx = pnorm(symbTx)
    sigRx  = pnorm(sigRx)

    # Alinha rotação de fase
    ind = np.arange(discard, min(len(sigRx), len(symbTx)) - discard)
    rot = np.mean(symbTx[ind] / sigRx[ind])
    sigRx = sigRx * rot

    # Decisão dura
    idx_rx = qam_demod(sigRx[ind], constSymb)
    idx_tx = qam_demod(symbTx[ind], constSymb)

    # Bits recebidos e transmitidos
    bitsRx = symb_to_bits(idx_rx, M)
    bitsTx = symb_to_bits(idx_tx, M)

    # BER
    BER = np.mean(bitsRx != bitsTx)
    num_erros = np.sum(bitsRx != bitsTx)

    # SNR
    noise = sigRx[ind] - symbTx[ind]
    SNR = np.mean(np.abs(symbTx[ind])**2) / np.mean(np.abs(noise)**2)
    SNR_dB = 10 * np.log10(SNR)

    return SNR_dB, BER, num_erros

In [ ]:
SNR, BER, erros = estimate_snr_ber(y_EQ[0], np.squeeze(symbTx_), M=64, constType='qam', discard=2000, SpS=4)
print('P_sig[opt] = %.2f dBm'%(10*np.log10(signal_power(sigCh)/1e-3)))
print('P_sig_eq[opt] = %.2f dBm'%(10*np.log10(signal_power(y_EQ[0])/1e-3)))
print('SNR[est] = %.2f dB \n'%(10*np.log10(SNR)))
print('Total de erros contados = %d  '%(erros))
print('BER = %.2e  '%(BER))

#### Conclusão

Em modulações de alta ordem, como 64-QAM, o uso direto do RDE tende a falhar na convergência, pois depende de uma estimativa precisa dos múltiplos raios da constelação. Para mitigar esse problema, a combinação CMA-to-RDE utiliza o algoritmo CMA como uma etapa de pré-convergência, alinhando os vetores iniciais do sinal, facilitando a atuação do RDE posterior. Esta abordagem resulta em uma constelação mais definida e uma convergência mais confiável.

A função cpr realiza a recuperação da fase da portadora (CPR), um passo crítico em receptores coerentes para compensar os efeitos do ruído de fase introduzido pelos lasers. Ela estima e remove a rotação de fase no sinal recebido, permitindo a correta reconstrução da constelação.